In [ ]:
# Practice Problems Day 18

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice the new code from today: making the targets by moving the text over by one, making the causal mask, and building a tiny GPT that writes. Plan on about 45 minutes. Training a model takes a minute or two on a laptop, so be patient when you get to Problem 2!

**Solutions** are in `HW-day18-solutions.ipynb` in this folder. Try every problem yourself first, then check your work.

**Reading:** Geron chapter 15, **Building an English-to-Spanish Transformer**.

GOALS:

1. Build the input and the targets for next character prediction, and the causal mask.
2. Write the `forward` of a tiny GPT.
3. See what a shorter context window does to the model.
4. Explain in words how a GPT writes text.

**How much of this should you write yourself?** The setup cell is given: the text, `encode` and `decode`, the transformer block with the mask, the training function and `generate` from the notes. Problem 1 gives you most of the code, you write the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 18 practice"
git push
```

---

In [ ]:
# RUN THIS. Everything from the notes that you do not need to write today.
import math
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

text = open("data/shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
char_to_id = {}
for i in range(vocab_size):
    char_to_id[chars[i]] = i

def encode(some_text):
    ids = []
    for c in some_text:
        ids.append(char_to_id[c])
    return ids

def decode(ids):
    pieces = []
    for i in ids:
        pieces.append(chars[i])
    return "".join(pieces)

def make_pieces(context_length):
    # cut the text into pieces of context_length + 1, then the last 10% is validation
    ids = torch.tensor(encode(text))
    n_pieces = len(ids) // (context_length + 1)
    pieces = ids[:n_pieces * (context_length + 1)].reshape(n_pieces, context_length + 1)
    X = pieces[:, :-1]
    y = pieces[:, 1:]
    n_train = int(0.9 * n_pieces)
    return X[:n_train], y[:n_train], X[n_train:], y[n_train:]

class TransformerBlock(nn.Module):
    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_numbers)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)

    def forward(self, X, mask=None):
        attention_out, weights = self.attention(X, X, X, attn_mask=mask)
        Z = self.norm1(X + attention_out)
        out = self.norm2(Z + self.feed_forward(Z))
        return out

def train_gpt(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    train_losses = []
    valid_losses = []
    for epoch in range(n_epochs):
        model.train()
        total = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            scores = model(X_batch)
            loss = loss_fn(scores.reshape(-1, vocab_size), y_batch.reshape(-1))
            loss.backward()
            optimizer.step()
            total = total + loss.item()
        train_losses.append(total / len(train_loader))
        model.eval()
        with torch.no_grad():
            scores = model(X_valid)
            valid_loss = loss_fn(scores.reshape(-1, vocab_size), y_valid.reshape(-1)).item()
        valid_losses.append(valid_loss)
        print("epoch", epoch + 1, "  train loss:", round(train_losses[-1], 3), "  valid loss:", round(valid_loss, 3))
    return train_losses, valid_losses

def generate(model, start, n_new, temperature=1.0):
    ids = encode(start)
    model.eval()
    with torch.no_grad():
        for step in range(n_new):
            context = torch.tensor([ids[-model.context_length:]])   # the model remembers its own context length
            scores = model(context)
            probs = torch.softmax(scores[0, -1] / temperature, dim=0)
            ids.append(torch.multinomial(probs, num_samples=1).item())
    return decode(ids)

print("number of different characters:", vocab_size)

**NOTE** Read through the functions defined above you can use them!!!


## Problem 1: The targets, the mask, and the forward pass

Let's start by building the three new pieces from today. These are the same steps as in the notes, so if you get stuck you can go back and look. But try it on your own first!

**What to copy, what to write:** Most of the code is given. Write the lines that the comments ask for.

**1a.** Remember that the text is its own label: the target is the input moved over by one character. Turn "to be or not" into ids with `encode`. Make the input (every id but the last) and the target (every id but the first). Print both with `decode` so you can check that the target really is the input moved over by one.

**1b.** Make the causal mask for 4 spots and print it. How many spots are True? What does True mean in this mask? (BEWARE: this is the one that is easy to get backwards!)

**1c.** Fill in the `forward` of the tiny GPT below. You need two things: make the mask, and then send `Z` through every block with a `for` loop. Then send 2 pieces of 64 characters through a brand new model and check the shape that comes out. Before you run it, make a guess! What should the three numbers in the shape be?

In [ ]:
# 1a
ids = encode("to be or not")
# Make the input: every id but the last. Call it x_ids.

# Make the target: every id but the first. Call it y_ids.

# Print decode(x_ids) and decode(y_ids) inside repr(...) so you can see the spaces


In [ ]:
# 1b
# Make the causal mask for 4 spots with torch.triu, and print it



In [ ]:
# 1c
class TinyGPT(nn.Module):

    def __init__(self, context_length, n_numbers=64, n_heads=4, n_blocks=2):
        super().__init__()
        self.context_length = context_length                           # generate() reads this
        self.embed = nn.Embedding(vocab_size, n_numbers)
        self.position_embed = nn.Embedding(context_length, n_numbers)
        self.blocks = nn.ModuleList()
        for i in range(n_blocks):
            self.blocks.append(TransformerBlock(n_numbers, n_heads))
        self.out = nn.Linear(n_numbers, vocab_size)

    def forward(self, X):
        n_spots = X.shape[1]
        spots = torch.arange(n_spots)
        # Make the causal mask for n_spots spots. Call it mask.

        Z = self.embed(X) + self.position_embed(spots)
        # Send Z through every block in self.blocks, with the mask (a for loop).

        scores = self.out(Z)
        return scores

X_train, y_train, X_valid, y_valid = make_pieces(context_length=64)
torch.manual_seed(42)
model = TinyGPT(context_length=64)
# Send the first 2 training pieces, X_train[:2], through model and print the shape.
# It should have 3 numbers: (pieces, spots, scores).


*Your answers here.*

---

## Problem 2: A shorter context window (steps outlined)

In the notes our tiny GPT could see 64 characters at once, and after 5 epochs the validation loss was **1.827**. The context window is how far back the model can look. So what happens if we make it smaller, and the model can only see 16 characters back? Let's try it and see!

**What to copy, what to write:** `make_pieces`, `train_gpt` and `generate` are in the setup cell, and `TinyGPT` is your class from Problem 1. Write the steps below yourself.

**2a.** Make the pieces with a context window of 16 characters and train a `TinyGPT` with `context_length=16` for 5 epochs. Is the validation loss better or worse than 1.827? Make a guess before you run it!

**2b.** Have your model write 300 characters starting with "JULIET:". Compare it to the text in the notes. Why would a model that can only see 16 characters back have a harder time? Think about what it can't see when it is in the middle of a line, eg. who is talking.

In [ ]:
# Step 1: make the pieces with make_pieces(context_length=16)

# Step 2: set torch.manual_seed(42), then make a TinyGPT with context_length=16

# Step 3: train it for 5 epochs with train_gpt
# It prints 5 lines, one training and one validation loss per epoch.


In [ ]:
# Step 4: set torch.manual_seed(0), then print 300 characters from "JULIET:" with generate


*Your answers here.*

---

## Problem 3: In words

Being able to explain this in plain words is a really useful skill. People are going to ask you how these tools work!

**3a.** A friend asks "how does ChatGPT write a whole answer?" Explain it to them using the words **next**, **probabilities** and **one at a time**. Use what we built today, not what you have heard about it.

**3b.** Chatbots sometimes write things that sound right but are false. Use what you learned today about how a GPT is trained and how it picks the next piece of text to explain why that can happen.

*Your answers here.*

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Let's peek inside the model from Problem 1. First train it for 5 epochs, like in the notes. Then send in "ROMEO:\nWhat is th" and look at the probabilities for the next character. `torch.topk(probs, 5)` gives back the 5 biggest probabilities and their ids. Which 5 characters does the model like best? Do they make sense?

In [ ]:
# your code here

*Your answers here.*